In [6]:
import pandas as pd

raw = pd.read_csv("teaching_prices_messy.csv")
print(raw.shape)
print(raw.dtypes)
print(raw.isna().sum())

(97, 6)
date          object
item          object
city          object
price         object
unit          object
data_label    object
dtype: object
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64


In [7]:
# Duplicate row
raw[raw.duplicated(keep=False)]

,date,item,city,price,unit,data_label
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
96,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [8]:
# Item names, and counts by item and city
print(raw["item"].value_counts())
print(raw.groupby(["item", "city"]).size())

item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64
item         city  
Oil B        City 1    12
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    12
             City 2    12
Sweetener D  City 1    12
             City 2    12
oil b        City 1     1
dtype: int64


In [9]:
# Everything on 5 Feb (the "oil b" row)
raw[raw["date"] == "2026-02-05"]

,date,item,city,price,unit,data_label
40,2026-02-05,oil b,City 1,108.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
41,2026-02-05,Staple A,City 2,114.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
42,2026-02-05,Oil B,City 1,318.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
43,2026-02-05,Oil B,City 2,324.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
44,2026-02-05,Pulse C,City 1,202.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
45,2026-02-05,Pulse C,City 2,208.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
46,2026-02-05,Sweetener D,City 1,161.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
47,2026-02-05,Sweetener D,City 2,167.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [10]:
# Prices that are blank, not numbers, or too big
num = pd.to_numeric(raw["price"], errors="coerce")
raw[num.isna() | (num > 500)]

,date,item,city,price,unit,data_label
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
17,2026-01-15,Staple A,City 2,"1,250",PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
30,2026-01-22,Sweetener D,City 1,9999,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [11]:
ai = raw.copy()

# Line 1
ai = ai.drop_duplicates()
print("Rows:", len(ai))

Rows: 96


In [12]:
# Line 2
ai["item"] = ai["item"].str.strip().str.title()
print(ai.groupby(["item", "city"]).size())

item         city  
Oil B        City 1    13
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    11
             City 2    12
Sweetener D  City 1    12
             City 2    12
dtype: int64


In [13]:
# Line 3
ai["price"] = ai["price"].str.replace(",", "").astype(float)
ai[ai["price"] > 500]

,date,item,city,price,unit,data_label
17,2026-01-15,Staple A,City 2,1250.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
30,2026-01-22,Sweetener D,City 1,9999.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [14]:
# Line 4
ai = ai[ai["price"] < 1000]
print("Rows:", len(ai))

Rows: 93


In [15]:
# Line 5: what it would fill the blank price with
print(round(pd.to_numeric(raw["price"].str.replace(",", ""), errors="coerce").mean(), 2))

315.46


In [16]:
# Lines 6 and 7
ai["date"] = pd.to_datetime(ai["date"])
ai = ai.drop(columns=["unit", "data_label"])
print(ai.columns.tolist())
print(ai["item"].value_counts())

['date', 'item', 'city', 'price']
item
Oil B          25
Pulse C        23
Sweetener D    23
Staple A       22
Name: count, dtype: int64


In [17]:
clean = raw.copy()
clean["flag"] = ""

clean = clean.drop_duplicates()

m = clean["item"] == "oil b"
clean.loc[m, "item"] = "Staple A"
clean.loc[m, "flag"] = "item was 'oil b'"

m = clean["price"].isna()
clean.loc[m, "flag"] = "price blank"

m = clean["price"] == "1,250"
clean.loc[m, "flag"] = "price was 1,250"
clean.loc[m, "price"] = None

m = clean["price"] == "9999"
clean.loc[m, "flag"] = "price was 9999"
clean.loc[m, "price"] = None

clean["price"] = pd.to_numeric(clean["price"])
clean["date"] = pd.to_datetime(clean["date"])

clean[clean["flag"] != ""]

,date,item,city,price,unit,data_label,flag
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,price blank
17,2026-01-15,Staple A,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,"price was 1,250"
30,2026-01-22,Sweetener D,City 1,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,price was 9999
40,2026-02-05,Staple A,City 1,108.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,item was 'oil b'


In [18]:
print("Rows:", len(raw), "->", len(clean))
print("Missing price:", raw["price"].isna().sum(), "->", clean["price"].isna().sum())
print("\nBefore:")
print(pd.to_numeric(raw["price"].str.replace(",", ""), errors="coerce").describe().round(2))
print("\nAfter:")
print(clean["price"].describe().round(2))

Rows: 97 -> 96
Missing price: 1 -> 3

Before:
count      96.00
mean      315.46
std      1007.56
min        99.50
25%       142.25
50%       200.00
75%       304.00
max      9999.00
Name: price, dtype: float64

After:
count     93.00
mean     203.61
std       79.92
min       99.50
25%      149.00
50%      200.00
75%      298.00
max      350.00
Name: price, dtype: float64


In [19]:
# Does every item appear the number of times I expect?
print(clean["item"].value_counts())
print(clean.groupby(["item", "city"]).size())

item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
Name: count, dtype: int64
item         city  
Oil B        City 1    12
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    12
             City 2    12
Sweetener D  City 1    12
             City 2    12
dtype: int64
